# Praca z tekstem w pandas — `.str`, regex i triki z `lambda`

**Problem:** pandas ma bogaty accessor `.str` pokrywający większość zadań tekstowych bez `lambda` — ale różnica między pokrewnymi metodami (`split` vs `partition`, `find` vs `index`, `replace` z regex vs bez) bywa nieoczywista, a sięgnięcie po `lambda`/`apply()` tam, gdzie `.str` by wystarczyło, kosztuje realną wydajność.

**Porównanie:**
- `.str.*` — wektoryzowane, bezpiecznie obsługują `NaN` (zwracają `NaN`, nie rzucają błędu), zwykle szybsze.
- `.apply(lambda x: ...)` — elastyczne, ale wolniejsze i **nie** obsługują `NaN` automatycznie — trzeba to ogarnąć samemu.

**Kiedy stosować `lambda`:** gdy logika łączy kilka kolumn naraz (`axis=1`), zależy od warunku, którego nie da się wyrazić jedną metodą `.str`, albo gdy potrzebna mapa/słownik zależny od wartości w innej kolumnie. W każdym innym przypadku najpierw szukaj gotowej metody `.str`.

## Setup

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "product_code": ["PRD-2026-00123", "PRD-2026-00456", "SVC-2025-00789", "PRD-2026-00234"],
    "raw_name": ["  Router TP-Link AC1200 ", "SWITCH netgear 8-port", "cable HDMI 2m", "  Access Point UBIQUITI  "],
    "email": ["jan.kowalski@firma.pl", "anna_nowak@firma.pl", "piotr.wisniewski@firma.pl", "ewa-kowal@firma.pl"],
    "tags": ["electronics,networking,sale", "electronics,networking", "cables,accessories", "electronics,wifi,new"],
    "file_path": ["data/region_north/report_2026.csv", "data/region_south/report_2026.csv",
                  "data/region_east/summary.csv", "data/region_west/report_2026_v2.csv"],
})
df

,product_code,raw_name,email,tags,file_path
0,PRD-2026-00123,Router TP-Link AC1200,jan.kowalski@firma.pl,"electronics,networking,sale",data/region_north/report_2026.csv
1,PRD-2026-00456,SWITCH netgear 8-port,anna_nowak@firma.pl,"electronics,networking",data/region_south/report_2026.csv
2,SVC-2025-00789,cable HDMI 2m,piotr.wisniewski@firma.pl,"cables,accessories",data/region_east/summary.csv
3,PRD-2026-00234,Access Point UBIQUITI,ewa-kowal@firma.pl,"electronics,wifi,new",data/region_west/report_2026_v2.csv


## Sekcja 1 — Czyszczenie: `strip`, wielkość liter, długość

**`Series.str.strip(to_strip=None)` - Usuwa znaki z początku i końca każdego elementu tekstowego w Series. to_strip = None jest domyślne i oznacza usuwanie białych znaków (spacje, tabulatory, znaki karetki, znaki nowej lini).**

`s.str.strip(to_strip=".")` - usunie kropki z początku i końca

`s.str.strip("abc")` - Będzie usuwał dowolne pojedyncze znaki należące do zbioru {a, b, c} z początku i końca tekstu.

In [2]:
df["raw_name"].str.strip() #czyści spacje/białe spacje na początku i końcu i początku tekstu
df["raw_name_left_strip"] = df["raw_name"].str.strip().str.lstrip("RScA") #lstrip() / rstrip() - usuwa spacje tylko z lewej lub prawej strony
df

,product_code,raw_name,email,tags,file_path,raw_name_left_strip
0,PRD-2026-00123,Router TP-Link AC1200,jan.kowalski@firma.pl,"electronics,networking,sale",data/region_north/report_2026.csv,outer TP-Link AC1200
1,PRD-2026-00456,SWITCH netgear 8-port,anna_nowak@firma.pl,"electronics,networking",data/region_south/report_2026.csv,WITCH netgear 8-port
2,SVC-2025-00789,cable HDMI 2m,piotr.wisniewski@firma.pl,"cables,accessories",data/region_east/summary.csv,able HDMI 2m
3,PRD-2026-00234,Access Point UBIQUITI,ewa-kowal@firma.pl,"electronics,wifi,new",data/region_west/report_2026_v2.csv,ess Point UBIQUITI


**Uwaga na `.str.title()`:** kapitalizuje literę po KAŻDYM znaku niebędącym literą, nie tylko po spacji — `"8-port"` staje się `"8-Port"`, a `"2m"` → `"2M"`. Wygląda niewinnie, ale przy nazwach z myślnikami/cyframi potrafi zaskoczyć.

In [3]:
clean = df["raw_name"].str.strip()
print(clean.str.lower()) #wszystko małymi literami
print(clean.str.upper())
print(clean.str.isupper()) #sprawdza czy cały tekst jest pisany wielkimi literami --> True/False
print(clean.str.title())      # uwaga: "8-port" -> "8-Port", "2m" -> "2M"
print(clean.str.istitle()) #sprawdza czy tekst jest w formacie title --> zwraca True/False
print(clean.str.swapcase()) #zamienia małe litery na wielkie a wielkie na małe
print(clean.str.capitalize()) # tylko pierwsza litera calego stringa, reszta lower
print(clean.str.len())

0    router tp-link ac1200
1    switch netgear 8-port
2            cable hdmi 2m
3    access point ubiquiti
Name: raw_name, dtype: object
0    ROUTER TP-LINK AC1200
1    SWITCH NETGEAR 8-PORT
2            CABLE HDMI 2M
3    ACCESS POINT UBIQUITI
Name: raw_name, dtype: object
0    False
1    False
2    False
3    False
Name: raw_name, dtype: bool
0    Router Tp-Link Ac1200
1    Switch Netgear 8-Port
2            Cable Hdmi 2M
3    Access Point Ubiquiti
Name: raw_name, dtype: object
0    False
1    False
2    False
3    False
Name: raw_name, dtype: bool
0    rOUTER tp-lINK ac1200
1    switch NETGEAR 8-PORT
2            CABLE hdmi 2M
3    aCCESS pOINT ubiquiti
Name: raw_name, dtype: object
0    Router tp-link ac1200
1    Switch netgear 8-port
2            Cable hdmi 2m
3    Access point ubiquiti
Name: raw_name, dtype: object
0    21
1    21
2    13
3    21
Name: raw_name, dtype: int64


## Sekcja 2 — Dzielenie tekstu: `split`

- bez argumentów `n=` — dzieli na wszystkie wystąpienia separatora, zwraca listy o różnej długości.
- `n=` — ogranicza liczbę podziałów (przydatne np. przy adresie e-mail, żeby rozdzielić tylko na pierwszej kropce).
- `rsplit` — dzieli od końca; z `n=1` daje "wszystko oprócz ostatniego fragmentu" + "ostatni fragment" — idealne do rozdzielenia ścieżki na katalog i nazwę pliku.
- `expand=True` — zamiast list w jednej kolumnie, od razu rozbija wynik na osobne kolumny `DataFrame`.

In [4]:
#Series.str.split(pat=None, n=-1, expand=False, regex=None)
df["product_code"].str.split("-")

0    [PRD, 2026, 00123]
1    [PRD, 2026, 00456]
2    [SVC, 2025, 00789]
3    [PRD, 2026, 00234]
Name: product_code, dtype: object

In [5]:
# n= ogranicza liczbe podzialow - tu tylko na pierwszej kropce w mailu
df["email"].str.split(".", n=1)

0        [jan, kowalski@firma.pl]
1          [anna_nowak@firma, pl]
2    [piotr, wisniewski@firma.pl]
3           [ewa-kowal@firma, pl]
Name: email, dtype: object

In [6]:
# rsplit z n=1 - katalog + nazwa pliku, licząc od KOŃCA ścieżki
df["file_path"].str.rsplit("/", n=1)

0      [data/region_north, report_2026.csv]
1      [data/region_south, report_2026.csv]
2           [data/region_east, summary.csv]
3    [data/region_west, report_2026_v2.csv]
Name: file_path, dtype: object

### Pułapka — `split(expand=True)` z różną liczbą fragmentów tworzy `NaN`

Jeśli liczba wystąpień separatora różni się między wierszami, brakujące kolumny wypełniają się `NaN` — nie ma żadnego ostrzeżenia, że któryś wiersz miał inny kształt niż pozostałe.

In [7]:
# expand=True - od razu jako osobne kolumny DataFrame, nie listy
df["product_code"].str.split("-", expand=True)


,0,1,2
0,PRD,2026,00123
1,PRD,2026,00456
2,SVC,2025,00789
3,PRD,2026,00234


In [8]:
df["product_code"].str.split("-").str[1] #str.split() zwraca listę wyników za pomocą str[x] możemy pobrać konkretny fragment po wybranym separatorze - str[1] - tekst o indeksie 1

0    2026
1    2026
2    2025
3    2026
Name: product_code, dtype: object

** `split()` z regexem: kilka możliwych separatorów naraz**

Gdy dane mają niespójne separatory (typowe przy ręcznie wprowadzanych danych — raz przecinek, raz średnik, raz spacja), wzorzec regex w `split()` obsługuje wszystkie naraz w jednym wywołaniu.

In [9]:
messy_separators = pd.Series(["a,b;c d", "x; y,z"])
messy_separators.str.split(r"[,; ]+")  # przecinek, średnik LUB spacja, jedno lub więcej pod rząd

0    [a, b, c, d]
1       [x, y, z]
dtype: object

## Sekcja 3 — Tekst przed/po separatorze: `partition` / `rpartition`

`partition()` dzieli **tylko na pierwszym** wystąpieniu separatora i zawsze zwraca dokładnie 3 kolumny: `(przed, separator, po)`. `rpartition()` robi to samo, ale od **ostatniego** wystąpienia. W przeciwieństwie do `split`, liczba kolumn wyniku jest zawsze stała — nie trzeba martwić się o `expand=True` ani o różną liczbę fragmentów (patrz też Pułapka 2).

In [10]:
df["email"].str.partition("@")

,0,1,2
0,jan.kowalski,@,firma.pl
1,anna_nowak,@,firma.pl
2,piotr.wisniewski,@,firma.pl
3,ewa-kowal,@,firma.pl


In [11]:
# rpartition na ścieżce pliku - dokładnie ten sam efekt co rsplit(n=1), inny zapis
df["file_path"].str.rpartition("/")

,0,1,2
0,data/region_north,/,report_2026.csv
1,data/region_south,/,report_2026.csv
2,data/region_east,/,summary.csv
3,data/region_west,/,report_2026_v2.csv


## Sekcja 4 — Wyciąganie fragmentów: `extract` / `extractall`

`extract()` z grupami regex (`(...)`)  zwraca jedno dopasowanie na wiersz — po jednej kolumnie na grupę. **Nazwane grupy** (`(?P<nazwa>...)`) od razu nadają czytelne nazwy kolumn zamiast `0, 1, 2`. `extractall()` zwraca **wszystkie** dopasowania w tekście (nie tylko pierwsze), z dodatkowym poziomem indeksu numerującym kolejne trafienia.

In [12]:
#df["product_code"].str.extract(r"WZORZEC")
df["product_code"].str.extract(r"(?P<prefix>[A-Z]+)-(?P<year>\d+)-(?P<number>\d+)")

,prefix,year,number
0,PRD,2026,00123
1,PRD,2026,00456
2,SVC,2025,00789
3,PRD,2026,00234


In [13]:
codes_in_text = pd.Series(["kod: A1, kod: B2, kod: C3"])
codes_in_text.str.extractall(r"kod: (?P<kod>\w\d)")

kod
  match    
0 0      A1
  1      B2
  2      C3

## Sekcja 5 — Tylko cyfry / tylko litery

Najprostsze podejście: regex `replace()` usuwający wszystko, co NIE pasuje do wzorca (`[^0-9]` = "nie-cyfra", `[0-9]` = "cyfra" do usunięcia).

In [14]:
#df["product_code"].str.extract(r"(\d+)") # wyciąga tylko cyfry
df["product_code"].str.extract(r"([A-Z]+)-(\d+)") # dzieli kolumnę na dwie, gdzie w jednej jest tylko tekst, a w drugiej tylko cyfry, dwa nawiasy () --> dwie kolumny

,0,1
0,PRD,2026
1,PRD,2026
2,SVC,2025
3,PRD,2026


In [15]:
print("Tylko cyfry:")
print(df["product_code"].str.replace(r"[^0-9]", "", regex=True))

print("\nTylko litery (usuwamy cyfry i myślniki):")
print(df["product_code"].str.replace(r"[^A-Za-z]", "", regex=True))

Tylko cyfry:
0    202600123
1    202600456
2    202500789
3    202600234
Name: product_code, dtype: object

Tylko litery (usuwamy cyfry i myślniki):
0    PRD
1    PRD
2    SVC
3    PRD
Name: product_code, dtype: object


## Sekcja 6 — Podmiana tekstu: `replace`

`Series.str.replace()` (podmiana fragmentu tekstu, z regexem lub bez) to inna metoda niż `Series.replace()` (podmiana **całych wartości**, jak w Sekcji o duplikatach z innych notatek) — łatwo je pomylić po samej nazwie.

In [16]:
clean = df["raw_name"].str.strip()

# str.replace - podmiana FRAGMENTU tekstu
print(clean.str.replace("TP-Link", "TPLink"))

# case=False - podmiana niezależna od wielkości liter (wymaga regex=True)
print(clean.str.replace("router", "ROUTER", case=False, regex=True))

# Series.replace (BEZ .str) - podmiana CAŁEJ wartości komórki, nie fragmentu
print(clean.replace({"cable HDMI 2m": "HDMI Cable 2m"}))

#podmiana znaków -,_,/ na pusty tekst
print(clean.str.replace(r"[-_/]", "", regex=True))

0     Router TPLink AC1200
1    SWITCH netgear 8-port
2            cable HDMI 2m
3    Access Point UBIQUITI
Name: raw_name, dtype: object
0    ROUTER TP-Link AC1200
1    SWITCH netgear 8-port
2            cable HDMI 2m
3    Access Point UBIQUITI
Name: raw_name, dtype: object
0    Router TP-Link AC1200
1    SWITCH netgear 8-port
2            HDMI Cable 2m
3    Access Point UBIQUITI
Name: raw_name, dtype: object
0     Router TPLink AC1200
1     SWITCH netgear 8port
2            cable HDMI 2m
3    Access Point UBIQUITI
Name: raw_name, dtype: object


## Sekcja 7 — Pozycja wystąpienia: `find`, `rfind`, `count`

`.str.find()` to odpowiednik Pythonowego `str.find()`, nie `str.index()` — przy braku dopasowania zwraca `-1` zamiast rzucać wyjątek. Bezpieczniejsze w kontekście wektoryzowanym, gdzie różne wiersze mogą, ale nie muszą zawierać szukanego fragmentu.

In [17]:
print(df["email"].str.find("@"))       # pozycja pierwszego wystąpienia
print(df["email"].str.count(r"\."))    # liczba wystąpień

print("\nRóżnica find vs Python index() przy braku dopasowania:")
print("pd.Series(['brak_znaku']).str.find('@') ->", pd.Series(["brak_znaku"]).str.find("@").tolist(), "(bez błędu)")
try:
    "brak_znaku".index("@")
except ValueError as e:
    print(f"czysty Python str.index('@') -> błąd: {e}")

0    12
1    10
2    16
3     9
Name: email, dtype: int64
0    2
1    1
2    2
3    1
Name: email, dtype: int64

Różnica find vs Python index() przy braku dopasowania:
pd.Series(['brak_znaku']).str.find('@') -> [-1] (bez błędu)
czysty Python str.index('@') -> błąd: substring not found


## Sekcja 8 — Sprawdzanie zawartości: `isalpha`, `isdigit`, `contains`, `startswith`

Cała rodzina `is*` sprawdza CAŁY string (nie pojedynczy znak) — `"abc123"` to ani `isalpha`, ani `isdigit`, bo miesza litery i cyfry.

In [18]:
words = pd.Series(["Warszawa", "12345", "abc123", "  ", "ABC"])
print("isalpha:", words.str.isalpha().tolist())
print("isdigit:", words.str.isdigit().tolist())
print("isalnum:", words.str.isalnum().tolist())  # litery LUB cyfry, mieszanka OK
print("isupper:", words.str.isupper().tolist())

isalpha: [True, False, False, False, True]
isdigit: [False, True, False, False, False]
isalnum: [True, True, True, False, True]
isupper: [False, False, False, False, True]


In [19]:
print(df["email"].str.contains("firma.pl"))       # uwaga: '.' w contains to też regex, patrz Pułapka 4
print(df["product_code"].str.startswith("PRD"))
print(df["file_path"].str.endswith(".csv"))

0    True
1    True
2    True
3    True
Name: email, dtype: bool
0     True
1     True
2    False
3     True
Name: product_code, dtype: bool
0    True
1    True
2    True
3    True
Name: file_path, dtype: bool


## Sekcja 9 — `removeprefix` / `removesuffix`

Bezpieczny odpowiednik ręcznego `str[len(prefix):]` — jeśli string NIE zaczyna się od podanego prefiksu, zwraca go bez zmian zamiast obcinać coś przypadkiem.

In [20]:
# 'SVC-2025-00789' nie zaczyna się od 'PRD-' -> zostaje bez zmian, bez błędu
df["product_code"].str.removeprefix("PRD-")

0        2026-00123
1        2026-00456
2    SVC-2025-00789
3        2026-00234
Name: product_code, dtype: object

In [21]:
pd.Series(["report_2026.csv", "summary.csv"]).str.removesuffix(".csv")

0    report_2026
1        summary
dtype: object

## Sekcja 10 — Łączenie tekstu: `str.cat`, `join`, `agg` + `", ".join`

In [22]:
# str.cat - sklejenie DWÓCH kolumn wiersz-po-wierszu
df["product_code"].str.cat(df["raw_name"].str.strip(), sep=" | ")

0    PRD-2026-00123 | Router TP-Link AC1200
1    PRD-2026-00456 | SWITCH netgear 8-port
2            SVC-2025-00789 | cable HDMI 2m
3    PRD-2026-00234 | Access Point UBIQUITI
Name: product_code, dtype: object

In [23]:
# .str.get(i) - dostęp do elementu listy po split, BEZ lambda (i=-1 to ostatni element)
split_tags = df["tags"].str.split(",")
print("Pierwszy tag:", split_tags.str.get(0).tolist())
print("Ostatni tag:", split_tags.str.get(-1).tolist())

# .str.join - sklejenie LISTY z powrotem w jeden string z innym separatorem
split_tags.str.join(" / ")

Pierwszy tag: ['electronics', 'electronics', 'cables', 'electronics']
Ostatni tag: ['sale', 'networking', 'accessories', 'new']


0    electronics / networking / sale
1           electronics / networking
2               cables / accessories
3           electronics / wifi / new
Name: tags, dtype: object

In [24]:
# agg + ", ".join po groupby - scalenie tekstów W OBRĘBIE grupy
sample = pd.DataFrame({"category": ["A", "A", "B", "B"], "tag": ["x", "y", "z", "w"]})
sample.groupby("category")["tag"].agg(", ".join)

category
A    x, y
B    z, w
Name: tag, dtype: object

## Sekcja 11 — Dopełnianie: `pad`, `ljust`, `rjust`, `center`, `zfill`

Przydatne przy wyrównywaniu tekstu do stałej szerokości (np. eksport do formatu o stałej długości pola) albo przy numerach/kodach wymagających zer wiodących.

In [25]:
cities = pd.Series(["Warszawa", "Lodz", "Gdansk"])

print(cities.str.ljust(12, fillchar="."))   # wyrównanie do LEWEJ, dopełnienie z prawej
print(cities.str.rjust(12, fillchar="."))   # wyrównanie do PRAWEJ, dopełnienie z lewej
print(cities.str.center(12, fillchar="."))  # wyśrodkowanie

0    Warszawa....
1    Lodz........
2    Gdansk......
dtype: object
0    ....Warszawa
1    ........Lodz
2    ......Gdansk
dtype: object
0    ..Warszawa..
1    ....Lodz....
2    ...Gdansk...
dtype: object


In [26]:
# zfill - dopełnienie zerami wiodącymi, typowe dla ID/kodów o stałej długości
order_numbers = pd.Series(["7", "42", "123"])
order_numbers.str.zfill(5)

0    00007
1    00042
2    00123
dtype: object

## Sekcja 12 — Wycinanie po pozycji: `slice`, `slice_replace`, `repeat`

`.str.slice()` to wektoryzowany odpowiednik zwykłego `string[a:b]` — przydatny, gdy struktura tekstu jest stała pozycyjnie (np. pierwsze 3 znaki to zawsze prefiks kategorii), a nie oddzielona separatorem (wtedy lepszy `split`/`partition`, Sekcje 2–3).

In [27]:
codes_fixed_width = pd.Series(["PRD20260123", "SVC20250789"])

print("Znaki 0-3 (prefiks):", codes_fixed_width.str.slice(0, 3).tolist())
print("Znaki 3-7 (rok):    ", codes_fixed_width.str.slice(3, 7).tolist())
print()

# slice_replace - podmiana fragmentu WSKAZANEGO POZYCYJNIE, nie przez dopasowanie wzorca
print(codes_fixed_width.str.slice_replace(0, 3, "XXX"))

Znaki 0-3 (prefiks): ['PRD', 'SVC']
Znaki 3-7 (rok):     ['2026', '2025']

0    XXX20260123
1    XXX20250789
dtype: object


In [28]:
# repeat - powtórzenie CAŁEGO stringa n razy (rzadkie, ale przydatne np. do separatorów wizualnych)
pd.Series(["ab", "cd"]).str.repeat(3)

0    ababab
1    cdcdcd
dtype: object

## Sekcja 13 — `match` vs `fullmatch` vs `contains`: trzy różne poziomy dopasowania

Wszystkie trzy przyjmują ten sam wzorzec regex, ale różnią się tym, JAK DUŻO tekstu musi pasować:
- `contains` — wzorzec może wystąpić GDZIEKOLWIEK w tekście.
- `match` — dopasowanie musi zaczynać się od POCZĄTKU tekstu (ale nie musi go w całości "skonsumować").
- `fullmatch` — CAŁY tekst musi dokładnie pasować do wzorca, od początku do końca.

In [29]:
codes_to_validate = pd.Series(["PRD-2026-00123", "invalid-code", "SVC-2025-00789"])
pattern = r"[A-Z]{3}-\d{4}-\d{5}"

print("contains:  ", codes_to_validate.str.contains(pattern, regex=True).tolist())
print("match:     ", codes_to_validate.str.match(pattern).tolist())
print("fullmatch: ", codes_to_validate.str.fullmatch(pattern).tolist())

contains:   [True, False, True]
match:      [True, False, True]
fullmatch:  [True, False, True]


In [30]:
# Różnica match vs fullmatch ujawnia się dopiero przy DODATKOWYM tekście na końcu
with_extra_suffix = pd.Series(["PRD-2026-00123-extra_dopisek"])

print(f"match:     {with_extra_suffix.str.match(pattern).tolist()}  (pasuje - dopasowanie zaczyna się poprawnie)")
print(f"fullmatch: {with_extra_suffix.str.fullmatch(pattern).tolist()}  (NIE pasuje - jest 'ogon' po dopasowaniu)")

match:     [True]  (pasuje - dopasowanie zaczyna się poprawnie)
fullmatch: [False]  (NIE pasuje - jest 'ogon' po dopasowaniu)


## Sekcja 14 — `translate()`: mapowanie znak-na-znak

Szybsza alternatywa dla łańcucha wielu `str.replace()`, gdy zamieniasz POJEDYNCZE ZNAKI na inne pojedyncze znaki — klasyczny przykład: usuwanie polskich znaków diakrytycznych (np. pod nazwy plików/identyfikatory bez ogonków).

In [31]:
polish_chars = "ąćęłńóśźżĄĆĘŁŃÓŚŹŻ"
ascii_chars = "acelnoszzACELNOSZZ"
translation_map = str.maketrans(polish_chars, ascii_chars)

cities_pl = pd.Series(["Łódź", "Gdańsk", "Wrocław"])
cities_pl.str.translate(translation_map)

0       Lodz
1     Gdansk
2    Wroclaw
dtype: object

## Sekcja 15 — Usuwanie duplikatów w tekście

In [ ]:
#usuwanie dwóch takich samych znaków obok siebie
df["tekst"].str.replace(r"(.)\1+", r"\1", regex=True)

#usunięcie wszsytkich znaków występujących więcej niż 1 raz
def remove_duplicates(text):
    return "".join(dict.fromkeys(text))

df["tekst"] = df["tekst"].apply(remove_duplicates)

#usuwanie powstarzających się spacji w środku tekstu
df["tekst"].str.replace(r"\s+", " ", regex=True)

## Podsumowanie

| Zadanie | Rozwiązanie |
|---|---|
| Usunięcie białych znaków z brzegów | `.str.strip()` / `lstrip()` / `rstrip()` |
| Wielkość liter | `.str.lower()` / `upper()` / `title()` (uwaga: kapitalizuje po `-`/cyfrach) / `capitalize()` |
| Podział na wszystkie fragmenty | `.str.split(sep)` |
| Podział z limitem / od końca | `.str.split(sep, n=)` / `.str.rsplit(sep, n=)` |
| Podział na osobne kolumny | `.str.split(sep, expand=True)` |
| Tekst przed/po PIERWSZYM separatorze | `.str.partition(sep)` |
| Tekst przed/po OSTATNIM separatorze | `.str.rpartition(sep)` |
| Wyciągnięcie fragmentu wg wzorca | `.str.extract(r"(?P<nazwa>...)")` |
| Wszystkie dopasowania wzorca | `.str.extractall(r"...")` |
| Tylko cyfry / tylko litery | `.str.replace(r"[^0-9]", "", regex=True)` / `r"[^A-Za-z]"` |
| Podmiana fragmentu tekstu | `.str.replace(stary, nowy, regex=..., case=...)` |
| Podmiana całej wartości komórki | `.replace({stara_wartość: nowa_wartość})` (bez `.str`) |
| Pozycja pierwszego wystąpienia (bezpieczna) | `.str.find(fragment)` (zwraca `-1`, nie rzuca błędu) |
| Liczba wystąpień | `.str.count(wzorzec)` |
| Czy CAŁY string to litery/cyfry | `.str.isalpha()` / `isdigit()` / `isalnum()` |
| Czy zawiera / zaczyna / kończy się na | `.str.contains()` / `startswith()` / `endswith()` |
| Bezpieczne usunięcie prefiksu/sufiksu | `.str.removeprefix()` / `.str.removesuffix()` |
| Sklejenie dwóch kolumn | `.str.cat(inna_kolumna, sep=...)` |
| Sklejenie listy (po split) w string | `.str.join(sep)` |
| Element listy po indeksie (bez lambda) | `.str.get(i)` (ujemny indeks też działa) |
| Scalenie tekstów w obrębie grupy | `.groupby("col").agg(", ".join)` |
| Logika łącząca kilka kolumn naraz | `.apply(lambda row: ..., axis=1)` — tu `lambda` jest uzasadniona |
| Wyrównanie do stałej szerokości | `.str.ljust()` / `rjust()` / `center()` |
| Zera wiodące (np. ID o stałej długości) | `.str.zfill(n)` |
| Fragment po POZYCJI (nie separatorze) | `.str.slice(a, b)` / `.str.slice_replace(a, b, nowy)` |
| Powtórzenie całego stringa n razy | `.str.repeat(n)` |
| Dopasowanie od początku / całego tekstu | `.str.match(wzorzec)` / `.str.fullmatch(wzorzec)` |
| Podmiana zależna od dopasowania (nie stały tekst) | `.str.replace(wzorzec, funkcja, regex=True)` |
| Podział po kilku możliwych separatorach naraz | `.str.split(r"[,; ]+")` |
| Opcjonalny fragment we wzorcu | `(?:...)?` w regexie |
| Mapowanie pojedynczych znaków (np. usuwanie diakrytyków) | `.str.translate(str.maketrans(...))` |
